In [ ]:
import os
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# Adjust this path if weblog.txt lives elsewhere
LOG_PATH = os.path.join(os.path.dirname(os.path.abspath(f'{os.getenv("HOME")}')), 'weblog.txt')
print('Log file exists:', os.path.exists(LOG_PATH))
print('Size (MB):', round(os.path.getsize(LOG_PATH) / 1_000_000, 1))

# Peek at the first 3 raw lines
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 2:
            break

# Re-define with corrected grouping (matches data_loader.py exactly)
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ '
    r'\[(?P<datetime>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<url>\S+) \S+" '
    r'(?P<status>\d{3}) (?P<size>\S+) '
    r'"(?P<referer>[^"]*)" '
    r'"(?P<user_agent>[^"]*)"'
)

# Test it on one line
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    sample_line = f.readline()

m = LOG_PATTERN.match(sample_line)
if m:
    print('Parsed fields:')
    for k, v in m.groupdict().items():
        print(f'  {k:12s}: {v}')
else:
    print('No match — check that LOG_PATH points to a valid Apache log')

#%%time
records = []
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        m = LOG_PATTERN.match(line)
        if m:
            d = m.groupdict()
            try:
                size = int(d['size']) if d['size'] != '-' else 0
            except ValueError:
                size = 0
            records.append({
                'ip':           d['ip'],
                'datetime_str': d['datetime'],
                'method':       d['method'],
                'url':          d['url'],
                'status':       int(d['status']),
                'size':         size,
                'referer':      d['referer'],
                'user_agent':   d['user_agent'],
            })

df_raw = pd.DataFrame(records)
print(f'Parsed {len(df_raw):,} log lines  ({len(df_raw.columns)} columns)')
df_raw.head(3)

# ── Timestamp parsing ────────────────────────────────────────────────────────
df = df_raw.copy()
df['timestamp'] = pd.to_datetime(
    df['datetime_str'],
    format='%d/%b/%Y:%H:%M:%S %z',
    errors='coerce',
    utc=True
)
df = df.dropna(subset=['timestamp'])  # drop unparseable lines

df['date']        = df['timestamp'].dt.date
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0 = Monday
df['month']       = df['timestamp'].dt.to_period('M').astype(str)

print('Date range:', df['timestamp'].min().date(), '→', df['timestamp'].max().date())
df[['timestamp', 'date', 'hour', 'day_of_week', 'month']].head(3)

# ── Static resource detection ────────────────────────────────────────────────
STATIC_EXTENSIONS = {
    '.css', '.js', '.png', '.jpg', '.jpeg', '.gif', '.ico',
    '.svg', '.woff', '.woff2', '.ttf', '.eot', '.map',
}

def is_static(url):
    lower = url.lower()
    if os.path.splitext(lower)[1] in STATIC_EXTENSIONS:
        return True
    if lower.startswith(('/css/', '/js/', '/fonts/', '/images/', '/static/', '/socket.io/')):
        return True
    return 'favicon' in lower

df['is_static'] = df['url'].apply(is_static)

# ── Bot detection ────────────────────────────────────────────────────────────
BOT_PATTERNS = re.compile(
    r'bot|crawler|spider|crawl|slurp|bingbot|googlebot|feedcrawler|'
    r'yandex|baidu|semrush|ahref|mj12|dotbot|petalbot', re.I
)

df['is_bot'] = df['user_agent'].apply(lambda ua: bool(BOT_PATTERNS.search(ua)))

# ── Status categories ────────────────────────────────────────────────────────
df['status_cat'] = pd.cut(
    df['status'],
    bins=[0, 199, 299, 399, 499, 599],
    labels=['1xx Info', '2xx Success', '3xx Redirect', '4xx Client Err', '5xx Server Err']
)

# ── Section (first URL segment) ───────────────────────────────────────────────
df['section'] = df['url'].apply(
    lambda u: '/' + u.strip('/').split('/')[0] if u != '/' else '/'
)

print(f'Total rows: {len(df):,}')
print(f'Static requests : {df["is_static"].sum():,} ({df["is_static"].mean():.1%})')
print(f'Bot requests    : {df["is_bot"].sum():,} ({df["is_bot"].mean():.1%})')
df[['url', 'status', 'status_cat', 'is_static', 'is_bot', 'section']].head(5)

# Column types and non-null counts at a glance
df.dtypes.to_frame('dtype').assign(non_null=df.count())

# Quick numeric summary of the most useful columns
df[['status', 'size', 'hour', 'day_of_week']].describe().round(1)

df_pages  = df[~df['is_static']]
df_humans = df_pages[~df_pages['is_bot']]

print(f'df        (all requests)   : {len(df):>8,} rows')
print(f'df_pages  (content only)   : {len(df_pages):>8,} rows  ({len(df_pages)/len(df):.1%} of all)')
print(f'df_humans (humans + pages) : {len(df_humans):>8,} rows  ({len(df_humans)/len(df):.1%} of all)')

total_requests  = len(df)
unique_visitors = df['ip'].nunique()
bandwidth_gb    = df['size'].sum() / 1e9
error_rate      = (df['status'] >= 400).mean()
bot_pct         = df['is_bot'].mean()

print(f'Total requests  : {total_requests:,}')
print(f'Unique IPs      : {unique_visitors:,}')
print(f'Bandwidth (GB)  : {bandwidth_gb:.2f}')
print(f'Error rate      : {error_rate:.1%}')
print(f'Bot traffic     : {bot_pct:.1%}')

daily = df_humans.groupby('date').size().reset_index(name='requests')


Log file exists: True
Size (MB): 6.3
192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET / HTTP/1.1" 200 523 "-" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"
192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET /css/ait.css HTTP/1.1" 200 7749 "https://huaxia.ai/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"
192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET /css/bootstrap.css HTTP/1.1" 200 145969 "https://huaxia.ai/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"
Parsed fields:
  ip          : 192.168.1.1
  datetime    : 17/Jul/2023:21:16:28 -0400
  method      : GET
  url         : /
  status      : 200
  size        : 523
  referer     : -
  user_agent  : Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0
Parsed 27,598 log lines  (8 columns)
Date range: 2023-07-18 → 2024-10-10


C:\Users\nizea\AppData\Local\Temp\ipykernel_33060\52432473.py:82: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0 = Monday
C:\Users\nizea\AppData\Local\Temp\ipykernel_33060\52432473.py:83: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month']       = df['timestamp'].dt.to_period('M').astype(str)


Total rows: 27,598
Static requests : 17,940 (65.0%)
Bot requests    : 1,880 (6.8%)
df        (all requests)   :   27,598 rows
df_pages  (content only)   :    9,658 rows  (35.0% of all)
df_humans (humans + pages) :    8,895 rows  (32.2% of all)
Total requests  : 27,598
Unique IPs      : 1,092
Bandwidth (GB)  : 0.27
Error rate      : 7.7%
Bot traffic     : 6.8%


NameError: name 'plt' is not defined